In [ ]:
import os
import glob
import random
import pandas as pd
import torch
from torch.utils.data import Dataset
import torchvision.transforms as transforms
from trdg.generators import GeneratorFromStrings

class OnTheFlyTRDGDataset(Dataset):
    def __init__(self, tsv_path, fonts_dir):
        self.df = pd.read_csv(
                    tsv_path, 
                    sep="\t", 
                    header=None, 
                    names=["ids", "langs", "texts"], 
                    quoting=3, 
                    on_bad_lines="skip"
                )
        self.df = self.df[self.df["id"] != "id"].reset_index(drop=True)
        
        self.all_fonts = glob.glob(os.path.join(fonts_dir, "*.ttf")) + glob.glob(os.path.join(fonts_dir, "*.otf"))
        if not self.all_fonts:
            raise FileNotFoundError(f"Tidak ada font ditemukan di folder {fonts_dir}")
            
        self.transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1), 
            transforms.Resize((64, 256)),               
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))         
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text_label = str(row["text"])
        
        skew_angle = random.randint(0, 1)  
        dist_type = 0 
        dist_ori = 0
        bg_type = 0  
        img_size = 64
        
        gen = GeneratorFromStrings(
            [text_label],
            fonts=[random_font],
            skewing_angle=skew_angle,
            random_skew=False, 
            distorsion_type=dist_type,
            distorsion_orientation=dist_ori,
            background_type=bg_type,
            size=img_size,
            fit=False,
        )
        
        img, _ = next(iter(gen))
        img_tensor = self.transform(img)
        return img_tensor, text_label

In [ ]:
from torch.utils.data import DataLoader

dataset_train = OnTheFlyTRDGDataset(tsv_path="corpus_2.tsv", fonts_dir="font")

train_loader = DataLoader(
    dataset_train, 
    batch_size=64,      
    shuffle=True,       
    num_workers=1,     
    pin_memory=False      
)

In [ ]:
import os
import string
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

all_chars = string.ascii_letters + string.digits + " ()?!-"

char_map = {"[BLANK]": 0}
for idx, char in enumerate(all_chars):
    char_map[char] = idx + 1

NUM_CLASSES = len(char_map)

def encode_labels(labels_list, char_map):
    targets = []
    target_lengths = []
    for label in labels_list:
        encoded = [char_map[c] for c in label if c in char_map]
        targets.extend(encoded)
        target_lengths.append(len(encoded))
    return torch.tensor(targets, dtype=torch.long), torch.tensor(
        target_lengths, dtype=torch.long
    )

class CRNNModel(nn.Module):

    def __init__(self, num_classes, hidden_size=256):
        super(CRNNModel, self).__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), 
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d((2, 1), stride=(2, 1)),  
            nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d((2, 1), stride=(2, 1)),
        )

        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, None))

        self.rnn = nn.LSTM(
            512, hidden_size, bidirectional=True, num_layers=2, batch_first=True
        )

        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        features = self.cnn(x)  
        features = self.adaptive_pool(features)  


        features = features.squeeze(2)  
        features = features.permute(
            0, 2, 1
        )  

        rnn_out, _ = self.rnn(features)  

        out = self.fc(rnn_out)  

        out = out.permute(1, 0, 2)
        return out


BATCH_SIZE = 16  
LR = 0.001
TOTAL_EPOCHS = 50
CHECKPOINT_PATH = "checkpoint_ocr_tuf.pt"

# Load Dataset Anda
dataset_train = OnTheFlyTRDGDataset(tsv_path="corpus_2.tsv", fonts_dir="fonts")
train_loader = DataLoader(
    dataset_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training akan berjalan di: {device}")

model = CRNNModel(num_classes=NUM_CLASSES).to(device)
criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = optim.Adam(model.parameters(), lr=LR)

start_epoch = 0
global_step = 0

if os.path.exists(CHECKPOINT_PATH):
    print(
        f"➔ Menemukan checkpoint lama di '{CHECKPOINT_PATH}'. Mencoba me-restore progres..."
    )
    checkpoint = torch.load(CHECKPOINT_PATH)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
    global_step = checkpoint["global_step"]
    print(f"✓ Berhasil melanjutkan training dari Epoch ke-{start_epoch}!")
else:
    print(
        "➔ Tidak ada checkpoint ditemukan. Memulai training dari awal (Epoch 0)."
    )

for epoch in range(start_epoch, TOTAL_EPOCHS):
    model.train()
    running_loss = 0.0

    print(f"\n=== Epoch {epoch+1}/{TOTAL_EPOCHS} ===")

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device)

        targets, target_lengths = encode_labels(labels, char_map)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)

        optimizer.zero_grad()


        outputs = model(images) 

        input_lengths = torch.full(
            size=(images.size(0),), fill_value=outputs.size(0), dtype=torch.long
        ).to(device)

        loss = criterion(
            outputs.log_softmax(2), targets, input_lengths, target_lengths
        )

        loss.backward()
        optimizer.step()

        global_step += 1
        running_loss += loss.item()

        if batch_idx % 50 == 0:
            print(
                f"Batch {batch_idx}/{len(train_loader)} - Loss Saat Ini: {loss.item():.4f}"
            )

    print(f"➔ Mengamankan progres Epoch {epoch+1}...")
    torch.save(
        {
            "epoch": epoch,
            "global_step": global_step,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
        },
        CHECKPOINT_PATH,
    )
    print(
        "Checkpoint tersimpan! "
    )

print("\n Training Selesai 100%!")